In [ ]:
# ============================================
# 1) Instalar dependências e pedir API keys
# ============================================
!pip install -q pandas requests openpyxl tqdm python-levenshtein

import getpass
TMDB_KEY = getpass.getpass("TMDb API key (v3): ")
OMDB_KEY = getpass.getpass("OMDb API key (enter para saltar): ").strip() or None

from google.colab import files
uploaded = files.upload()

FNAME = list(uploaded.keys())[0]   # nome do ficheiro carregado
print("Ficheiro carregado:", FNAME)

# ============================================
# 2) Imports e config
# ============================================
import pandas as pd, requests, time, re
from tqdm import tqdm
from Levenshtein import ratio as lev_ratio

TMDB_BASE = "https://api.themoviedb.org/3"
OMDB_BASE = "https://www.omdbapi.com/"
TMDB_IMG_BASE = "https://image.tmdb.org/t/p/w500"

SLEEP = 0.25   # pausa pequena entre calls TMDb

# Nome das colunas esperadas no Excel
COL_MOVIE_TITLE   = "Movie Title"
COL_DATE          = "Date"
COL_LEAD_DIR      = "Lead Director"
COL_LEAD_ACTOR    = "Lead Actor"
COL_DIR_BDATE     = "Lead Director Birthdate"
COL_ACT_BDATE     = "Lead Actor Birthdate"
COL_DIR_NAT       = "Director Nationality"
COL_ACT_NAT       = "Actor Nationality"
COL_BUDGET        = "Budget"
COL_POSTER        = "Poster_URL"

# ============================================
# 3) Helpers HTTP + normalização texto
# ============================================
def http_get(url, params=None, retries=3, timeout=20):
    """GET com pequenos retries e backoff simples."""
    back = 0.6
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(back)
                back *= 2
                continue
            return r
        except requests.RequestException:
            time.sleep(back)
            back *= 2
    return None

_clean_re = re.compile(r"[^a-z0-9 ]+")

def norm_title(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s*\(\d{4}\)$","",s)
    s = _clean_re.sub(" ", s)
    s = re.sub(r"\s+"," ", s)
    return s

def get_row_year(row):
    """Inferir ano a partir de 'Year' ou 'Date' se existir."""
    y = None
    if "Year" in row and pd.notna(row["Year"]):
        try:
            y = int(row["Year"])
        except:
            y = None
    if y is None and COL_DATE in row and pd.notna(row[COL_DATE]):
        try:
            y = pd.to_datetime(row[COL_DATE], errors="coerce").year
            if pd.isna(y):
                y = None
        except:
            y = None
    return int(y) if y is not None else None

# ============================================
# 4) Caches simples em memória
# ============================================
movie_cache = {}   # (title_norm, year) -> dict com budget, poster
person_cache = {}  # name_norm -> dict com birthdate, nationality

# ============================================
# 5) TMDb: procurar filme com fuzzy title + ano
# ============================================
def tmdb_search_movie_candidates(title, year=None):
    """Procura candidatos de filme no TMDb por título (+/- ano)."""
    if not title:
        return []
    title = str(title)
    base_params = {
        "api_key": TMDB_KEY,
        "query": title,
        "include_adult": "false"
    }

    years = []
    if year is not None:
        years = [year, year-1, year+1]
    else:
        years = [None]

    seen = set()
    candidates = []

    for y in years:
        params = dict(base_params)
        if y is not None:
            params["year"] = int(y)
        r = http_get(f"{TMDB_BASE}/search/movie", params=params)
        if not r or r.status_code >= 400:
            continue
        results = r.json().get("results") or []
        for m in results:
            mid = m.get("id")
            if mid not in seen:
                seen.add(mid)
                candidates.append(m)
        time.sleep(SLEEP)
    return candidates

def score_movie_candidate(item, want_title, want_year=None):
    """Score combinado: popularidade + similaridade de título + penalização de ano."""
    vc   = item.get("vote_count", 0) or 0
    pop  = item.get("popularity", 0.0) or 0.0
    rdat = item.get("release_date") or ""
    yr   = int(rdat[:4]) if len(rdat) >= 4 and rdat[:4].isdigit() else None
    year_pen = abs(yr - want_year) if (want_year is not None and yr is not None) else 0

    sim = lev_ratio(norm_title(item.get("title","")), norm_title(want_title))

    score = vc*10 + pop*2 + sim*15 - year_pen*3
    return score, sim

def pick_best_tmdb_movie(title, year=None, sim_threshold=0.60):
    cands = tmdb_search_movie_candidates(title, year)
    if not cands:
        return None

    scored = []
    for it in cands:
        sc, sim = score_movie_candidate(it, title, year)
        scored.append((sc, sim, it))

    scored.sort(key=lambda x: x[0], reverse=True)
    best_sc, best_sim, best_item = scored[0]
    if best_sim < sim_threshold:
        return None
    return best_item

# ============================================
# 6) TMDb + OMDb: budget e poster
# ============================================
def get_movie_budget_and_poster(title, year=None):
    """
    Devolve (budget, poster_url) usando:
      1) TMDb details
      2) OMDb fallback (se tiveres OMDB_KEY)
    Usa cache para não repetir chamadas.
    """
    if not title:
        return None, None

    key = (norm_title(title), year)
    if key in movie_cache:
        return movie_cache[key]["budget"], movie_cache[key]["poster_url"]

    # 1) escolher filme no TMDb
    pick = pick_best_tmdb_movie(title, year)
    budget = None
    poster_url = None

    imdb_id = None

    if pick:
        mid = pick.get("id")
        # detalhes do filme
        r = http_get(f"{TMDB_BASE}/movie/{mid}", {"api_key": TMDB_KEY})
        if r and r.status_code < 400:
            j = r.json()
            budget = j.get("budget")
            poster_path = j.get("poster_path")
            if poster_path:
                poster_url = f"{TMDB_IMG_BASE}{poster_path}"
        time.sleep(SLEEP)

        # external_ids para obter IMDb id
        r2 = http_get(f"{TMDB_BASE}/movie/{mid}/external_ids", {"api_key": TMDB_KEY})
        if r2 and r2.status_code < 400:
            imdb_id = r2.json().get("imdb_id")
        time.sleep(SLEEP)

    # 2) Fallback OMDb (se necessário e se houver key)
    if OMDB_KEY and (budget in (None, 0) or poster_url is None):
        params = {"apikey": OMDB_KEY, "type": "movie"}
        # se tivermos imdb_id usamos, senão tentamos por título
        if imdb_id:
            params["i"] = imdb_id
        else:
            params["t"] = title
            if year:
                params["y"] = int(year)
        r = http_get(OMDB_BASE, params=params)
        if r and r.status_code < 400:
            data = r.json()
            if data.get("Response") == "True":
                if budget in (None, 0):
                    raw_budget = data.get("BoxOffice") or data.get("Budget")
                    if raw_budget and raw_budget != "N/A":
                        try:
                            clean = raw_budget.replace("$","").replace(",","")
                            budget = int(clean)
                        except:
                            pass
                if poster_url is None:
                    poster = data.get("Poster")
                    if poster and poster != "N/A":
                        poster_url = poster
        time.sleep(SLEEP)

    movie_cache[key] = {
        "budget": budget,
        "poster_url": poster_url
    }
    return budget, poster_url

# ============================================
# 7) TMDb: pessoas (ator / realizador)
# ============================================
def search_person_tmdb(name):
    """Procura pessoa por nome no TMDb e devolve o mais popular."""
    if not isinstance(name, str) or not name.strip():
        return None
    name_norm = name.strip().lower()
    if name_norm in person_cache:
        return person_cache[name_norm]["raw"]

    params = {"api_key": TMDB_KEY, "query": name, "include_adult": "false"}
    r = http_get(f"{TMDB_BASE}/search/person", params=params)
    if not r or r.status_code >= 400:
        return None
    results = r.json().get("results") or []
    if not results:
        return None

    # escolher mais popular
    best = max(results, key=lambda x: x.get("popularity", 0) or 0)
    person_cache[name_norm] = {"raw": best}  # vamos completar depois se for preciso
    time.sleep(SLEEP)
    return best

def get_person_birthdate_and_nationality(name):
    """
    Devolve (birthdate, nationality) para uma pessoa,
    usando /search/person + /person/{id}.
    Nationality aproximada a partir de place_of_birth (último segmento).
    """
    if not isinstance(name, str) or not name.strip():
        return None, None

    key = name.strip().lower()
    # se já tivermos info detalhada em cache:
    if key in person_cache and "birthdate" in person_cache[key]:
        return person_cache[key]["birthdate"], person_cache[key]["nationality"]

    base = search_person_tmdb(name)
    if not base:
        return None, None

    pid = base.get("id")
    if not pid:
        return None, None

    r = http_get(f"{TMDB_BASE}/person/{pid}", {"api_key": TMDB_KEY})
    if not r or r.status_code >= 400:
        return None, None

    j = r.json()
    birthdate = j.get("birthday")   # 'YYYY-MM-DD' ou None
    pob = j.get("place_of_birth")   # ex: "New York City, New York, USA"
    nationality = None
    if pob:
        parts = [p.strip() for p in pob.split(",") if p.strip()]
        if parts:
            nationality = parts[-1]

    # guardar cache
    person_cache[key]["birthdate"]   = birthdate
    person_cache[key]["nationality"] = nationality

    time.sleep(SLEEP)
    return birthdate, nationality

# ============================================
# 8) Lógica para decidir se atualizamos a data
# ============================================
def choose_better_birthdate(existing, candidate):
    """
    Regras:
      - se candidate vazio -> mantém existing
      - se existing vazio  -> usa candidate
      - se ano for diferente -> mantém existing (não confiamos)
      - se ano igual e candidate tem mais detalhe -> usa candidate
    """
    if candidate is None or str(candidate).strip() == "":
        return existing

    if existing is None or str(existing).strip() == "":
        return candidate

    # tentar ler datas
    try:
        e_dt = pd.to_datetime(existing, errors="coerce")
    except Exception:
        e_dt = pd.NaT
    try:
        c_dt = pd.to_datetime(candidate, errors="coerce")
    except Exception:
        c_dt = pd.NaT

    e_year = e_dt.year if pd.notna(e_dt) else None
    c_year = c_dt.year if pd.notna(c_dt) else None

    # se não conseguimos ano da candidate, não arriscamos
    if c_year is None:
        return existing

    # se existing tem ano e é diferente -> mantém existing
    if e_year is not None and e_year != c_year:
        return existing

    # anos iguais → escolhemos a string "mais detalhada"
    existing_str = str(existing)
    candidate_str = str(candidate)
    if len(candidate_str) > len(existing_str):
        return candidate
    else:
        return existing

# ============================================
# 9) Ler Excel e garantir colunas
# ============================================
df = pd.read_excel(FNAME)

for col in [COL_DIR_BDATE, COL_ACT_BDATE, COL_DIR_NAT, COL_ACT_NAT, COL_BUDGET, COL_POSTER]:
    if col not in df.columns:
        df[col] = pd.NA

# não vamos tocar noutras datas; só usamos Date para inferir ano
if COL_DATE in df.columns:
    df["_ReleaseYear_tmp"] = pd.to_datetime(df[COL_DATE], errors="coerce").dt.year
else:
    df["_ReleaseYear_tmp"] = pd.NA

# ============================================
# 10) Enriquecer filmes: Budget + Poster_URL
# ============================================
print("A enriquecer filmes com Budget & Poster_URL...")
movie_keys = df[[COL_MOVIE_TITLE, "_ReleaseYear_tmp"]].drop_duplicates()

movie_to_budget = {}
movie_to_poster = {}

for _, row in tqdm(movie_keys.iterrows(), total=len(movie_keys)):
    title = row[COL_MOVIE_TITLE]
    year  = row["_ReleaseYear_tmp"]
    if not isinstance(title, str) or not title.strip():
        continue

    b, p = get_movie_budget_and_poster(title, year)
    movie_to_budget[(title, year)] = b
    movie_to_poster[(title, year)] = p

# aplicar ao df sem apagar valores já existentes
for i, row in df.iterrows():
    key = (row[COL_MOVIE_TITLE], row["_ReleaseYear_tmp"])
    if pd.isna(df.at[i, COL_BUDGET]):
        df.at[i, COL_BUDGET] = movie_to_budget.get(key, df.at[i, COL_BUDGET])
    if pd.isna(df.at[i, COL_POSTER]):
        df.at[i, COL_POSTER] = movie_to_poster.get(key, df.at[i, COL_POSTER])

# ============================================
# 11) Enriquecer Realizadores: Birthdate + Nationality
# ============================================
print("A enriquecer realizadores (datas + nacionalidade)...")
directors = df[COL_LEAD_DIR].dropna().drop_duplicates()

dir_to_birth = {}
dir_to_nat   = {}

for name in tqdm(directors):
    birth, nat = get_person_birthdate_and_nationality(name)
    dir_to_birth[name] = birth
    dir_to_nat[name]   = nat

for i, row in df.iterrows():
    name = row[COL_LEAD_DIR]

    # birthdate refinada
    cand_birth = dir_to_birth.get(name)
    existing_birth = df.at[i, COL_DIR_BDATE]
    df.at[i, COL_DIR_BDATE] = choose_better_birthdate(existing_birth, cand_birth)

    # nationality só se estiver vazio
    if (pd.isna(df.at[i, COL_DIR_NAT]) or str(df.at[i, COL_DIR_NAT]).strip() == "") and dir_to_nat.get(name):
        df.at[i, COL_DIR_NAT] = dir_to_nat[name]

# ============================================
# 12) Enriquecer Atores: Birthdate + Nationality
# ============================================
print("A enriquecer atores (datas + nacionalidade)...")
actors = df[COL_LEAD_ACTOR].dropna().drop_duplicates()

act_to_birth = {}
act_to_nat   = {}

for name in tqdm(actors):
    birth, nat = get_person_birthdate_and_nationality(name)
    act_to_birth[name] = birth
    act_to_nat[name]   = nat

for i, row in df.iterrows():
    name = row[COL_LEAD_ACTOR]

    cand_birth = act_to_birth.get(name)
    existing_birth = df.at[i, COL_ACT_BDATE]
    df.at[i, COL_ACT_BDATE] = choose_better_birthdate(existing_birth, cand_birth)

    if (pd.isna(df.at[i, COL_ACT_NAT]) or str(df.at[i, COL_ACT_NAT]).strip() == "") and act_to_nat.get(name):
        df.at[i, COL_ACT_NAT] = act_to_nat[name]

# ============================================
# 13) Limpeza e guardar
# ============================================
if "_ReleaseYear_tmp" in df.columns:
    df.drop(columns=["_ReleaseYear_tmp"], inplace=True)

OUTFILE = f"enriched_{FNAME}"
df.to_excel(OUTFILE, index=False)
print("✅ Enriquecimento concluído →", OUTFILE)

files.download(OUTFILE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.8 MB/s eta 0:00:00
TMDb API key (v3): ··········
OMDb API key (enter para saltar): ··········


Saving Final Data Set.xlsx to Final Data Set.xlsx
Ficheiro carregado: Final Data Set.xlsx
A enriquecer filmes com Budget & Poster_URL...


100%|██████████| 848/848 [22:27<00:00,  1.59s/it]


A enriquecer realizadores (datas + nacionalidade)...


100%|██████████| 669/669 [06:19<00:00,  1.76it/s]


A enriquecer atores (datas + nacionalidade)...


100%|██████████| 616/616 [05:35<00:00,  1.83it/s]


✅ Enriquecimento concluído → enriched_Final Data Set.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>